# 02 - Data Cleaning: Preparing Tour de France Data

This notebook focuses on cleaning and standardizing the raw data collected in the web scraping phase.

## Objectives
- Load raw scraped data
- Standardize names, dates, and times
- Handle missing data
- Convert relative times to absolute timestamps
- Save cleaned data for analysis

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import json

## Setup Data Paths

In [ ]:
# Define paths
raw_data_dir = Path('data/raw')
cleaned_data_dir = Path('data/cleaned')
cleaned_data_dir.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {raw_data_dir.absolute()}")
print(f"Cleaned data directory: {cleaned_data_dir.absolute()}")

## Part 1: Load Raw Data

We'll load the data we scraped in the previous notebook.

In [ ]:
# Load Wikipedia races data
wiki_races_df = pd.read_csv(raw_data_dir / 'wikipedia_races.csv')
print(f"Loaded {len(wiki_races_df)} Wikipedia race records")
print("\nFirst few rows:")
wiki_races_df.head()

In [ ]:
# Load Tour de France stages data
tour_stages_df = pd.read_csv(raw_data_dir / 'tour_stages.csv')
print(f"Loaded {len(tour_stages_df)} Tour de France stage records")
print("\nFirst few rows:")
tour_stages_df.head()

## Part 2: Data Exploration and Quality Check

Let's examine the data quality and identify issues.

In [ ]:
# Check for missing values in Wikipedia races
print("Missing values in Wikipedia races data:")
print(wiki_races_df.isnull().sum())
print("\nData types:")
print(wiki_races_df.dtypes)

In [ ]:
# Check for missing values in Tour stages
print("Missing values in Tour stages data:")
print(tour_stages_df.isnull().sum())
print("\nData types:")
print(tour_stages_df.dtypes)
print("\nBasic statistics:")
print(tour_stages_df.describe())

## Part 3: Clean Wikipedia Races Data

We'll standardize the Wikipedia races data.

In [ ]:
def clean_wikipedia_races(df):
    """
    Clean and standardize Wikipedia races data.
    """
    df_clean = df.copy()
    
    # Standardize column names (lowercase with underscores)
    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')
    
    # Remove leading/trailing whitespace from string columns
    for col in df_clean.select_dtypes(include=['object']).columns:
        df_clean[col] = df_clean[col].str.strip()
    
    # Replace 'N/A' with actual NaN
    df_clean.replace('N/A', np.nan, inplace=True)
    
    # Drop rows with missing race names (critical field)
    df_clean = df_clean.dropna(subset=['name'])
    
    # Fill missing country/type with 'Unknown'
    df_clean['country'] = df_clean['country'].fillna('Unknown')
    df_clean['type'] = df_clean['type'].fillna('Unknown')
    
    # Remove duplicates
    df_clean = df_clean.drop_duplicates(subset=['name'], keep='first')
    
    print(f"Cleaned Wikipedia races: {len(df)} → {len(df_clean)} records")
    return df_clean

wiki_races_clean = clean_wikipedia_races(wiki_races_df)
wiki_races_clean.head()

## Part 4: Clean Tour de France Stages Data

This is where we'll do more intensive cleaning:
- Standardize dates
- Convert time strings to proper format
- Handle missing data
- Add calculated fields

In [ ]:
def parse_time_to_seconds(time_str):
    """
    Convert time string (HH:MM:SS) to total seconds.
    """
    if pd.isna(time_str):
        return np.nan
    
    parts = str(time_str).split(':')
    if len(parts) == 3:
        hours, minutes, seconds = map(int, parts)
        return hours * 3600 + minutes * 60 + seconds
    return np.nan

def seconds_to_time_str(seconds):
    """
    Convert seconds to HH:MM:SS format.
    """
    if pd.isna(seconds):
        return np.nan
    
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

In [ ]:
def clean_tour_stages(df):
    """
    Clean and standardize Tour de France stages data.
    """
    df_clean = df.copy()
    
    # Standardize column names
    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')
    
    # Convert date to datetime
    df_clean['date'] = pd.to_datetime(df_clean['date'])
    
    # Remove leading/trailing whitespace from string columns
    for col in df_clean.select_dtypes(include=['object']).columns:
        df_clean[col] = df_clean[col].str.strip()
    
    # Ensure distance is numeric
    df_clean['distance_km'] = pd.to_numeric(df_clean['distance_km'], errors='coerce')
    
    # Convert time to seconds for easier calculations
    df_clean['time_seconds'] = df_clean['time'].apply(parse_time_to_seconds)
    
    # Calculate average speed (km/h)
    df_clean['avg_speed_kmh'] = np.where(
        df_clean['time_seconds'] > 0,
        (df_clean['distance_km'] / df_clean['time_seconds']) * 3600,
        np.nan
    )
    
    # Round to 2 decimal places
    df_clean['avg_speed_kmh'] = df_clean['avg_speed_kmh'].round(2)
    
    # Standardize stage type categories
    stage_type_mapping = {
        'flat': 'Flat',
        'hilly': 'Hilly',
        'mountain': 'Mountain',
        'time trial': 'Time Trial',
        'tt': 'Time Trial'
    }
    df_clean['type'] = df_clean['type'].str.lower().map(stage_type_mapping).fillna(df_clean['type'])
    
    # Drop rows with missing critical data
    df_clean = df_clean.dropna(subset=['year', 'stage', 'date'])
    
    # Sort by year and stage
    df_clean = df_clean.sort_values(['year', 'stage']).reset_index(drop=True)
    
    print(f"Cleaned Tour stages: {len(df)} → {len(df_clean)} records")
    return df_clean

tour_stages_clean = clean_tour_stages(tour_stages_df)
tour_stages_clean.head()

## Part 5: Data Quality Verification

Let's verify our cleaned data.

In [ ]:
# Check cleaned Tour stages data
print("Cleaned Tour stages data info:")
print(tour_stages_clean.info())
print("\nMissing values:")
print(tour_stages_clean.isnull().sum())
print("\nSummary statistics:")
print(tour_stages_clean[['distance_km', 'time_seconds', 'avg_speed_kmh']].describe())

In [ ]:
# Display sample of cleaned data with new calculated fields
print("Sample cleaned data:")
tour_stages_clean[['year', 'stage', 'date', 'distance_km', 'type', 'winner', 'avg_speed_kmh']].head(10)

## Part 6: Save Cleaned Data

Finally, we'll save our cleaned data for analysis.

In [ ]:
# Save cleaned Wikipedia races
wiki_races_clean.to_csv(cleaned_data_dir / 'wikipedia_races_clean.csv', index=False)
wiki_races_clean.to_json(cleaned_data_dir / 'wikipedia_races_clean.json', orient='records', indent=2)

print(f"Saved cleaned Wikipedia races data: {len(wiki_races_clean)} records")

In [ ]:
# Save cleaned Tour stages
# Convert datetime to string for JSON serialization
tour_stages_for_json = tour_stages_clean.copy()
tour_stages_for_json['date'] = tour_stages_for_json['date'].dt.strftime('%Y-%m-%d')

tour_stages_clean.to_csv(cleaned_data_dir / 'tour_stages_clean.csv', index=False)
tour_stages_for_json.to_json(cleaned_data_dir / 'tour_stages_clean.json', orient='records', indent=2)

print(f"Saved cleaned Tour stages data: {len(tour_stages_clean)} records")

## Summary

In this notebook, we:
1. ✓ Loaded raw scraped data from CSV files
2. ✓ Explored data quality and identified issues
3. ✓ Standardized column names and data types
4. ✓ Handled missing data appropriately
5. ✓ Converted time strings to seconds for calculations
6. ✓ Added calculated fields (average speed)
7. ✓ Saved cleaned data in CSV and JSON formats

### Data Cleaning Highlights
- Converted dates to proper datetime format
- Parsed time strings and converted to seconds
- Calculated average speeds from distance and time
- Standardized categorical values (stage types)
- Removed duplicates and handled missing values

### Output Files
- `data/cleaned/wikipedia_races_clean.csv` - Cleaned race information
- `data/cleaned/wikipedia_races_clean.json` - Same data in JSON
- `data/cleaned/tour_stages_clean.csv` - Cleaned Tour de France stages
- `data/cleaned/tour_stages_clean.json` - Same data in JSON

### Next Steps
Proceed to `03_data_analysis.ipynb` to analyze the cleaned data and answer interesting questions about Tour de France performance.